In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Path to your trained model weights (e.g., best.pt or last.pt)
model_path = "/content/best.pt"
model = YOLO(model_path)

In [ ]:
video_path = "/content/lying1.mp4"

results = model.predict(source=video_path,
    show=True,
    save=True,
    imgsz=640)
# Predicted video will be saved in 'runs/pose/predict'

WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/482) /content/lying1.mp4: 384x640 1 cow, 568.6ms
video 1/1 (frame 2/482) /content/lying1.mp4: 384x640 2 cows, 277.2ms
video 1/1 (frame 3/482) /content/lying1.mp4: 384x640 2 cows, 149.2ms
video 1/1 (frame 4/482) /content/lying1.mp4: 384x640 2 cows, 137.6ms
video 1/1 (frame 5/482) /content/lying1.mp4: 384x640 2 cows, 133.6ms
video 1/1 (frame 6/482) /content/lying1.

In [ ]:
import os
import shutil

# Path to the 'runs' directory
runs_dir = "runs"

# Check if the directory exists before attempting to delete
if os.path.exists(runs_dir):
    shutil.rmtree(runs_dir)
    print(f"Directory '{runs_dir}' removed successfully.")
else:
    print(f"Directory '{runs_dir}' does not exist.")

Directory 'runs' removed successfully.


In [ ]:
import json
import uuid

# Define keypoint names based on the example output provided by the user.
# This assumes the first 10 keypoints from the model's output correspond to these names.
keypoint_names = [
    "right-ear", "nose", "left-ear", "neck", "left-front-hoof",
    "right-front-hoof", "hip", "left-back-hoof", "right-back-hoof", "tail"
]

all_predictions = []

# The 'results' variable is available from the previously executed model.predict() call.
# It's a list of Results objects, typically one per image. We process the first image's results.
if results and len(results) > 0:
    first_image_results = results[0]

    # Loop through each detected instance (e.g., each cow) in the image
    num_detections = len(first_image_results.boxes)

    for i in range(num_detections):
        # Extract bounding box information for the current detection
        x_center, y_center, width, height = first_image_results.boxes.xywh[i].tolist()
        confidence = first_image_results.boxes.conf[i].item()
        class_id = int(first_image_results.boxes.cls[i].item())
        class_name = first_image_results.names[class_id] # Get class name (e.g., 'cow')

        # Prepare the list of keypoints for the current detection
        current_keypoints = []

        # keypoints.xy has shape (num_detections, num_keypoints, 2)
        # keypoints.conf has shape (num_detections, num_keypoints)

        # Determine how many keypoints to process, limiting to the number of defined names
        num_keypoints_predicted = len(first_image_results.keypoints.xy[i])
        num_keypoints_to_output = min(len(keypoint_names), num_keypoints_predicted)

        for kp_idx in range(num_keypoints_to_output):
            kp_x, kp_y = first_image_results.keypoints.xy[i][kp_idx].tolist()
            kp_conf = first_image_results.keypoints.conf[i][kp_idx].item()

            current_keypoints.append({
                "x": round(kp_x),
                "y": round(kp_y),
                "confidence": round(kp_conf, 3),
                "class_id": kp_idx, # Assuming class_id for keypoints is their index
                "class": keypoint_names[kp_idx]
            })

        # Construct the prediction dictionary for the current detected object
        prediction = {
            "x": round(x_center, 1),
            "y": round(y_center, 1),
            "width": round(width),
            "height": round(height),
            "confidence": round(confidence, 3),
            "class": class_name,
            "class_id": class_id,
            "detection_id": str(uuid.uuid4()), # Generate a unique ID
            "keypoints": current_keypoints
        }
        all_predictions.append(prediction)

# Final JSON structure
output_json = {"predictions": all_predictions}

# Print the JSON output in a human-readable format
print(json.dumps(output_json, indent=2))


{
  "predictions": [
    {
      "x": 368.5,
      "y": 328.6,
      "width": 677,
      "height": 599,
      "confidence": 0.855,
      "class": "cow",
      "class_id": 0,
      "detection_id": "bc890d63-d3a1-4691-89b0-68c222d8c9df",
      "keypoints": [
        {
          "x": 149,
          "y": 160,
          "confidence": 0.782,
          "class_id": 0,
          "class": "right-ear"
        },
        {
          "x": 59,
          "y": 263,
          "confidence": 0.404,
          "class_id": 1,
          "class": "nose"
        },
        {
          "x": 341,
          "y": 180,
          "confidence": 0.7,
          "class_id": 2,
          "class": "left-ear"
        },
        {
          "x": 213,
          "y": 245,
          "confidence": 0.733,
          "class_id": 3,
          "class": "neck"
        },
        {
          "x": 320,
          "y": 529,
          "confidence": 0.772,
          "class_id": 4,
          "class": "left-front-hoof"
        },
        {
 

# Evaluation

In [ ]:
from ultralytics import YOLO
model = YOLO("best.pt")

In [ ]:
metrics = model.val(data='/content/cow_pose_dataset_small.yaml')

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n-pose summary (fused): 82 layers, 3,452,315 parameters, 0 gradients, 9.9 GFLOPs



FileNotFoundError: Dataset '/content/cow_pose_dataset_small.yaml' images not found, missing path '/kaggle/working/cow_pose_dataset_small/images/val'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'